# Modelo para el Analisis de factores que influyen en la popularidad de los videojuegos

Institucion: Escuela Politecnica Nacional EPN  
Materia: Metodos Numericos  
Semestre: 2026-A  
Fecha de entrega: 27 de mayo de 2026  
Estudiantes:  
* Ricardo Cisneros  
* Hugo Coyago  
* Jair Paez  

---  

# Parte 1: Creacion, Unificacion y Estructuracion del Dataset

Para garantizar la replicabilidad absoluta de este proyecto cientifico, la primera etapa consiste en reconstruir el dataset unificado completo directamente desde los datos crudos extraidos de la API de Steam y SteamSpy, los cuales estan almacenados en la carpeta 01_Datos/Datos crudos/steam-insights-main.

### Fuentes de Informacion Utilizadas
1. games.csv: Metadatos de la tienda oficial de Steam, incluyendo nombre, fecha de lanzamiento, tipo de producto y otros detalles.
2. reviews.csv: Estadisticas de valoraciones y resenas de usuarios publicas.
3. steamspy_insights.csv: Datos de consumo, precios en centavos y rangos de compradores estimativos.

### Formulas Matematicas de Estimacion Continua
Para transformar las categorias cualitativas o discretas en variables cuantitativas continuas, las cuales son indispensables para aplicar regresion y minimos cuadrados, implementamos dos ecuaciones clave:

1. Puntaje de Aprobacion Continuo A_R:
   $$A_R = \frac{\text{Resenas Positivas}}{\text{Resenas Positivas} + \text{Resenas Negativas}}$$
   Genera una escala de [0.0, 1.0] que representa el porcentaje real de satisfaccion del jugador.

2. Ventas Estimadas Continuas E_O:
   Dado que SteamSpy reporta las ventas en rangos categoricos [L, U] que representan el Limite Inferior y el Limite Superior respectivamente, calculamos su punto medio aritmetico:
   $$E_O = \frac{L + U}{2}$$
   Ejemplo: Para un rango de 10M a 20M de copias, calculamos un punto medio continuo de 15M.

### Variables Cuantitativas Clave Seleccionadas
Para garantizar la coherencia cientifica del modelo y evitar multicolinealidad o dependencias lineales perfectas, nos enfocaremos en las siguientes variables cuantitativas que existen en el dataset consolidado:

#### Variable Objetivo Eje Y
* estimated_owners: Ventas estimadas continuas obtenidas mediante el punto medio del rango de compradores.

#### Variables Predictoras Independientes Eje X
* price_usd: Precio de venta actual normalizado en dolares continuos.
* steamspy_initial_price: Precio de venta original del videojuego antes de aplicar descuentos.
* steamspy_discount: Porcentaje de descuento o rebaja activa en la tienda.
* approval_ratio: Proporcionalidad de aprobacion continua de los usuarios en una escala de cero a uno.
* reviews_total: Volumen total de opiniones y calificaciones registradas por la comunidad de Steam.
* concurrent_users_yesterday: Usuarios activos simultaneos registrados el dia anterior.
* is_free: Indicador binario discreto de gratuidad del producto donde uno es gratis y cero es de pago.
* language_count: Cantidad total de idiomas de localizacion en los que esta disponible el software.
* category_count: Cantidad total de caracteristicas tecnicas y modos de juego que ofrece el videojuego.
* release_year: Ano de lanzamiento numerico extraido de la fecha de publicacion oficial.

In [ ]:
import os
import csv

directorio_base = ".."
juegos_csv = os.path.join(directorio_base, "01_Datos", "Datos crudos", "steam-insights-main", "games", "games.csv")
resenas_csv = os.path.join(directorio_base, "01_Datos", "Datos crudos", "steam-insights-main", "reviews", "reviews.csv")
steamspy_csv = os.path.join(directorio_base, "01_Datos", "Datos crudos", "steam-insights-main", "steamspy_insights", "steamspy_insights.csv")
categorias_csv = os.path.join(directorio_base, "01_Datos", "Datos crudos", "steam-insights-main", "categories", "categories.csv")
salida_datos_unificados_crudos = os.path.join(directorio_base, "01_Datos", "steam_raw_unified_all_columns.csv")

def analizar_rango_duenos(cadena_duenos):
    if not cadena_duenos or '..' not in cadena_duenos:
        return ""
    partes = cadena_duenos.split('..')
    if len(partes) == 2:
        try:
            limite_inferior = int(partes[0].replace(',', '').strip())
            limite_superior = int(partes[1].replace(',', '').strip())
            return (limite_inferior + limite_superior) // 2
        except ValueError:
            return ""
    return ""

def extraer_ano_lanzamiento(fecha_str):
    if not fecha_str or fecha_str == "\\N":
        return ""
    # Buscar un grupo de 4 digitos consecutivos que representen el ano
    import re
    coincidencia = re.search(r'\b(19\d\d|20\d\d)\b', fecha_str)
    if coincidencia:
        return coincidencia.group(1)
    return ""

print("Cargando categorias...")
datos_categorias = {}
with open(categorias_csv, mode='r', encoding='utf-8') as archivo_cat:
    lector = csv.DictReader(archivo_cat)
    for fila in lector:
        id_app = fila.get("app_id")
        if id_app:
            datos_categorias[id_app] = datos_categorias.get(id_app, 0) + 1

print("Cargando juegos...")
datos_juegos = {}
with open(juegos_csv, mode='r', encoding='utf-8') as archivo_juegos:
    lector = csv.DictReader(archivo_juegos)
    for fila in lector:
        id_aplicacion = fila.get("app_id")
        if not id_aplicacion:
            continue
        
        idiomas_str = fila.get("languages", "").strip()
        cant_idiomas = 0
        if idiomas_str and idiomas_str != "\\N":
            cant_idiomas = len([lang.strip() for lang in idiomas_str.split(",") if lang.strip()])

        es_gratis_str = fila.get("is_free", "").strip().lower()
        es_gratis_num = ""
        if es_gratis_str == "true":
            es_gratis_num = 1
        elif es_gratis_str == "false":
            es_gratis_num = 0

        datos_juegos[id_aplicacion] = {
            "name": fila.get("name", ""),
            "release_date": fila.get("release_date", ""),
            "release_year": extraer_ano_lanzamiento(fila.get("release_date", "")),
            "is_free": es_gratis_num,
            "price_overview": fila.get("price_overview", ""),
            "games_languages": fila.get("languages", ""),
            "language_count": cant_idiomas,
            "type": fila.get("type", "")
        }

print("Cargando reseñas...")
datos_resenas = {}
with open(resenas_csv, mode='r', encoding='utf-8') as archivo_resenas:
    lector = csv.DictReader(archivo_resenas)
    for fila in lector:
        id_aplicacion = fila.get("app_id")
        if not id_aplicacion:
            continue
        datos_resenas[id_aplicacion] = {
            "review_score": fila.get("review_score", ""),
            "review_score_description": fila.get("review_score_description", ""),
            "reviews_positive": fila.get("positive", ""),
            "reviews_negative": fila.get("negative", ""),
            "reviews_total": fila.get("total", ""),
            "metacritic_score": fila.get("metacritic_score", ""),
            "reviews_text": fila.get("reviews", ""),
            "recommendations": fila.get("recommendations", ""),
            "steamspy_user_score": fila.get("steamspy_user_score", ""),
            "steamspy_score_rank": fila.get("steamspy_score_rank", ""),
            "reviews_steamspy_positive": fila.get("steamspy_positive", ""),
            "reviews_steamspy_negative": fila.get("steamspy_negative", "")
        }

print("Cargando SteamSpy...")
datos_steamspy = {}
todos_los_ids_aplicacion = set(datos_juegos.keys()) | set(datos_resenas.keys())
with open(steamspy_csv, mode='r', encoding='utf-8') as archivo_steamspy:
    lector = csv.DictReader(archivo_steamspy)
    for fila in lector:
        id_aplicacion = fila.get("app_id")
        if not id_aplicacion:
            continue
        datos_steamspy[id_aplicacion] = fila
        todos_los_ids_aplicacion.add(id_aplicacion)

print("Unificando y realizando transformaciones cuantitativas...")
todos_los_datos_fusionados = []
for id_aplicacion in sorted(list(todos_los_ids_aplicacion), key=lambda x: int(x) if x.isdigit() else 9999999):
    juego = datos_juegos.get(id_aplicacion, {
        "name": "", "release_date": "", "release_year": "", "is_free": "", "price_overview": "",
        "games_languages": "", "language_count": "", "type": ""
    })
    resena = datos_resenas.get(id_aplicacion, {
        "review_score": "", "review_score_description": "", "reviews_positive": "", "reviews_negative": "",
        "reviews_total": "", "metacritic_score": "", "reviews_text": "", "recommendations": "",
        "steamspy_user_score": "", "steamspy_score_rank": "", "reviews_steamspy_positive": "", "reviews_steamspy_negative": ""
    })
    spy = datos_steamspy.get(id_aplicacion, {
        "developer": "", "publisher": "", "owners_range": "", "concurrent_users_yesterday": "",
        "playtime_average_forever": "", "playtime_average_2weeks": "", "playtime_median_forever": "",
        "playtime_median_2weeks": "", "price": "", "initial_price": "", "discount": "", "languages": "", "genres": ""
    })

    precio_crudo = spy.get("price", "")
    precio_dolares = ""
    if precio_crudo and precio_crudo != "\\N":
        try:
            precio_dolares = round(int(precio_crudo) / 100.0, 2)
        except ValueError:
            pass

    precio_inicial_crudo = spy.get("initial_price", "")
    precio_inicial_dolares = ""
    if precio_inicial_crudo and precio_inicial_crudo != "\\N":
        try:
            precio_inicial_dolares = round(int(precio_inicial_crudo) / 100.0, 2)
        except ValueError:
            pass

    positivas_crudas = resena.get("reviews_positive", "")
    negativas_crudas = resena.get("reviews_negative", "")
    proporcion_aprobacion = ""
    if positivas_crudas and negativas_crudas:
        try:
            positivas = int(positivas_crudas)
            negativas = int(negativas_crudas)
            if positivas + negativas > 0:
                proporcion_aprobacion = round(positivas / (positivas + negativas), 4)
        except ValueError:
            pass

    duenos_crudo = spy.get("owners_range", "")
    duenos_estimados = analizar_rango_duenos(duenos_crudo)
    cant_categorias = datos_categorias.get(id_aplicacion, 0)
    
    fila_fusionada = {
        "app_id": id_aplicacion,
        "name": juego["name"],
        "release_date": juego["release_date"],
        "release_year": juego["release_year"],
        "is_free": juego["is_free"],
        "price_overview": juego["price_overview"],
        "games_languages": juego["games_languages"],
        "language_count": juego["language_count"],
        "category_count": cant_categorias,
        "type": juego["type"],
        "developer": spy.get("developer", ""),
        "publisher": spy.get("publisher", ""),
        "owners_range": spy.get("owners_range", ""),
        "concurrent_users_yesterday": spy.get("concurrent_users_yesterday", ""),
        "playtime_average_forever": spy.get("playtime_average_forever", ""),
        "playtime_average_2weeks": spy.get("playtime_average_2weeks", ""),
        "playtime_median_forever": spy.get("playtime_median_forever", ""),
        "playtime_median_2weeks": spy.get("playtime_median_2weeks", ""),
        "steamspy_price": spy.get("price", ""),
        "steamspy_initial_price": spy.get("initial_price", ""),
        "steamspy_discount": spy.get("discount", ""),
        "steamspy_languages": spy.get("languages", ""),
        "genres": spy.get("genres", ""),
        "review_score": resena["review_score"],
        "review_score_description": resena["review_score_description"],
        "reviews_positive": resena["reviews_positive"],
        "reviews_negative": resena["reviews_negative"],
        "reviews_total": resena["reviews_total"],
        "metacritic_score": resena["metacritic_score"],
        "reviews_text": resena["reviews_text"],
        "recommendations": resena["recommendations"],
        "steamspy_user_score": resena["steamspy_user_score"],
        "steamspy_score_rank": resena["steamspy_score_rank"],
        "reviews_steamspy_positive": resena["reviews_steamspy_positive"],
        "reviews_steamspy_negative": resena["reviews_steamspy_negative"],
        "price_usd": precio_dolares,
        "steamspy_initial_price_usd": precio_inicial_dolares,
        "approval_ratio": proporcion_aprobacion,
        "estimated_owners": duenos_estimados
    }
    todos_los_datos_fusionados.append(fila_fusionada)

encabezados = [
    "app_id", "name", "release_date", "release_year", "is_free", "price_overview", "games_languages",
    "language_count", "category_count", "type",
    "developer", "publisher", "owners_range", "concurrent_users_yesterday",
    "playtime_average_forever", "playtime_average_2weeks", "playtime_median_forever", "playtime_median_2weeks",
    "steamspy_price", "steamspy_initial_price", "steamspy_discount", "steamspy_languages", "genres",
    "review_score", "review_score_description", "reviews_positive", "reviews_negative", "reviews_total",
    "metacritic_score", "reviews_text", "recommendations", "steamspy_user_score", "steamspy_score_rank",
    "reviews_steamspy_positive", "reviews_steamspy_negative",
    "price_usd", "steamspy_initial_price_usd", "approval_ratio", "estimated_owners"
]

with open(salida_datos_unificados_crudos, mode='w', newline='', encoding='utf-8') as archivo_salida:
    escritor = csv.DictWriter(archivo_salida, fieldnames=encabezados)
    escritor.writeheader()
    escritor.writerows(todos_los_datos_fusionados)

print("Consolidacion completada con exito")

---  
## Parte 2: Limpieza de Datos y Ajuste de Curvas